# 🧠 DB-SLM: Instant Streaming Inference with Trained Model Weights

**Candidate:** Bilal Javed | Adept Tech Solutions

**Features:**
- ⚡ **Real-Time Token-by-Token Streaming** (`TextStreamer` prints responses live like ChatGPT)
- 🚀 **Fast KV-Caching Enabled** (`use_cache=True` runs generation in ~1–2 seconds on T4 GPU)
- 💾 **Google Drive Mounting** (Loads `db_slm_adapter` directly from Google Drive)

**Runtime:** Select `Runtime > Change runtime type > T4 GPU`

## Step 1: Mount Google Drive, Install Dependencies & Load Model

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
import os, zipfile
from pathlib import Path

print('Mounting Google Drive...')
drive.mount('/content/drive')

# 2. Install required inference packages
!pip install -q -U transformers peft bitsandbytes accelerate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TextStreamer
from peft import PeftModel

BASE_MODEL = 'Qwen/Qwen2.5-Coder-7B-Instruct'

# Check for adapter in Google Drive or local content
drive_zip = '/content/drive/MyDrive/db_slm_adapter.zip'
drive_folder = '/content/drive/MyDrive/db_slm_adapter'
local_dir = '/content/db_slm_adapter'

ADAPTER_DIR = local_dir
if Path(drive_folder).exists():
    ADAPTER_DIR = drive_folder
    print(f'✅ Found adapter folder in Google Drive: {drive_folder}')
elif Path(drive_zip).exists():
    print(f'Extracting adapter from Google Drive ({drive_zip})...')
    with zipfile.ZipFile(drive_zip, 'r') as zf:
        zf.extractall('/content/')
    ADAPTER_DIR = local_dir
elif Path('/content/db_slm_adapter.zip').exists():
    print('Extracting from uploaded zip in /content/...')
    !unzip -q /content/db_slm_adapter.zip -d /content/
    ADAPTER_DIR = local_dir

print(f'\nLoading base model: {BASE_MODEL} in 4-bit (T4 GPU)...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token is not None else tokenizer.pad_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

# Attach trained LoRA adapter weights
if Path(ADAPTER_DIR).exists() and (Path(ADAPTER_DIR) / 'adapter_model.safetensors').exists():
    print(f'Attaching fine-tuned LoRA adapter from: {ADAPTER_DIR}...')
    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    print('✅ Fine-tuned LoRA weights successfully loaded!')
else:
    model = base_model
    print('⚠️ Adapter folder not found. Running base model without adapter.')

SYSTEM_PROMPT = """You are an AI assistant trained on two real relational databases.

DATABASE 1 — Online Retail (UK e-commerce, Dec 2010 – Dec 2011):
- 4,372 customers across 38 countries (Domestic_UK, EU_Wholesale, International).
- 1,124 products, 25,900 invoices (14.8% cancellation rate), 54,873 line items.
- Confirmed order revenue (is_cancelled=0): £1,276,568.60 (avg £100.52 per order).
- Top revenue country: United Kingdom (£685,023), then Germany, France, EIRE, Spain.

DATABASE 2 — MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):
- 100 ICU patients (43F/57M, mean anchor age 61.8 years, 31 deceased).
- 275 hospital admissions with 15 deaths (5.5% in-hospital mortality rate).
- 140 ICU stays (average length of stay: 3.68 days).
- 107,727 lab results (37.4% abnormal), 18,087 prescriptions, 668,862 chartevents.
- seq_num=1 in diagnoses_icd denotes primary diagnosis.

CROSS-DATABASE: patient_customer_bridge (100 links) connects customer_id to subject_id.

Answer all queries accurately and specifically using this trained knowledge."""

# Set up Live Token-by-Token Streamer
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

def ask_stream(query, max_new_tokens=300):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': query}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    
    print(f'❓ Question: {query}\n' + '='*70 + '\n🤖 Response:')
    with torch.no_grad():
        model.generate(
            **inputs,
            streamer=streamer,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            use_cache=True,      # Fast KV-Cache generation (takes ~1-2s)
            pad_token_id=tokenizer.eos_token_id
        )
    print('='*70)

print('\n🚀 System is ready for fast live streaming queries!')

## Step 2: Query the Model (Live Streaming Output)

In [ ]:
# ── Type any question below and run this cell ──────────────────────────────────
ask_stream('What is the difference between invoice_items and products tables?')

### Suggested Questions to Try:
- `ask_stream('Explain the schema and table relationships of the Online Retail database in one line.')`
- `ask_stream('What is the difference between invoice_items and products tables?')`
- `ask_stream('What is the in-hospital mortality rate in MIMIC and how is it calculated?')`
- `ask_stream('Which country generates the highest retail revenue and why?')`
- `ask_stream('What does hospital_expire_flag mean in admissions?')`
- `ask_stream('How are retail customers and hospital patients linked in this system?')`